In [1]:
import dspy
from typing import List, Optional, Literal
from pydantic import BaseModel,Field

In [2]:
from openai import OpenAI
import os
import dspy

lm = dspy.LM(
    model="openai/gpt-5.2",
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)

### POLICY EXTRACTION ###

In [3]:
class ExtractedPolicy(BaseModel):
    # Core extraction
    policy_statement: str = Field(
        "A concise, self-contained summary of the policy commitment. "
        "MUST be directly supported by verbatim_text. "
        "Rewrite the text into a clear, active-voice statement."
    )
    
    verbatim_text: str = Field(
        "CRITICAL: Exact text from source document that supports this policy (2-3 sentences minimum). "
        "This will be programmatically verified against the source. Do not paraphrase."
    )
    
    # Hierarchy (NEW - replaces policy_group_name)
    policy_type: Literal["parent", "sub", "individual"] = Field(
        description=(
            "'parent': Umbrella program/plan with sub-policies listed below it in the document. "
            "'sub': Specific policy explicitly listed under a parent program. "
            "'individual': Standalone policy with no parent-child relationships."
        )
    )
    
    parent_policy_name: Optional[str] = Field(
        default=None,
        description="For 'sub' type only: exact name of the parent policy/program as written in document."
    )
    
    # Context
    section_header: str = Field(
        "The section or subsection heading this policy appears under. Copy exactly as written."
    )
    #probably don't need this
    sector: str = Field(
        "Primary climate sector"
    )

    

    extraction_rationale: str = Field(
        "Why this qualifies as a policy. Note any flags, vagueness, or edge cases."
    )

In [ ]:
'''
# Soundness indicators
    has_quantifiable_target: str = Field(
        "Indicate 'Yes' or 'No'. If 'Yes', provide details (e.g., '40% reduction by 2030')."
    )
    has_timeline: str = Field(
        "Indicate 'Yes' or 'No'. If 'Yes', provide details (e.g., 'by 2030')."
    )
    has_binding_mechanism: str = Field(
        "Indicate 'Yes' or 'No'. If 'Yes', provide details (e.g., 'legal mandate', 'ordinance')."
    )
    has_spatial_specificity: str = Field(
        "Indicate 'Yes' or 'No'. If 'Yes', provide details (e.g., 'citywide', 'district-level')."
    )
    has_implementation_pathway: str = Field(
        "Indicate 'Yes' or 'No'. 'Yes' if policy specifies HOW it will be achieved "
        "(named programs, responsible entities, funding, intermediate steps)."
    )'''

In [4]:
class DocumentMetadata(BaseModel):
    country: str 
    
    state_or_province: Optional[str]
    city: Optional[str] 


In [5]:
class PolicyExtractionSignature(dspy.Signature):
    """
    Extract climate policies from document text.
    
    CRITICAL: Extract ONLY information explicitly present in the document.
    Every policy MUST have verbatim text that can be verified.
    
    DEFINITION OF A POLICY
    A policy is a STATED COMMITMENT by a governing body to achieve a defined 
    outcome through deliberate action, resource allocation, or regulatory change.
    
    A policy is NOT:
    - Background information or problem descriptions
    - Statements of current conditions
    - Aspirations without any specified action
    - Descriptions of what other actors might do
    
    WHAT MAKES SOMETHING EXTRACTABLE
    Extract a statement as a policy if it contains AT LEAST ONE of:
    
    1. QUANTIFIABLE TARGET: Numbers with units and/or deadlines
    2. BINDING MECHANISM: Legal or regulatory force
    3. SPECIFIC INTERVENTION: Named program, technology, or action
    4. RESOURCE ALLOCATION: Committed funding or investment
    
    DO NOT EXTRACT
    - Pure context or problem statements
    - Current state descriptions
    - Process descriptions without commitments
    - Vague aspirations without concrete anchors
    
    ═══════════════════════════════════════════════════════════════════════
    HIERARCHY CLASSIFICATION
    ═══════════════════════════════════════════════════════════════════════
    
    Determine policy_type based on DOCUMENT STRUCTURE:
    
    ┌─────────────────────────────────────────────────────────────────────┐
    │ PARENT POLICY                                                       │
    │                                                                     │
    │ A policy is 'parent' when:                                         │
    │ • It introduces an action/program/initiative with a name           │
    │ • Multiple specific sub-items are listed beneath it                │
    │ • Sub-items are labeled (A, B, C) or (1, 2, 3)                    │
    │ • The parent describes overarching goal/scope                      │
    │                                                                     │
    │ For parent policies:                                               │
    │   policy_type = "parent"                                           │
    │   parent_policy_name = None                                        │
    │   policy_statement = Summary of the overarching action             │
    │   verbatim_text = Introductory text for the action group           │
    └─────────────────────────────────────────────────────────────────────┘
    
    ┌─────────────────────────────────────────────────────────────────────┐
    │ SUB POLICY                                                          │
    │                                                                     │
    │ A policy is 'sub' when:                                            │
    │ • It appears as an item in a lettered/numbered list                │
    │ • Listed under a named parent action/initiative                    │
    │ • Labeled with A., B., C. or 1., 2., 3.                           │
    │                                                                     │
    │ For sub policies:                                                  │
    │   policy_type = "sub"                                              │
    │   parent_policy_name = Exact name of parent action                 │
    │   policy_statement = The specific sub-item commitment              │
    │   verbatim_text = Text of this specific list item                  │
    └─────────────────────────────────────────────────────────────────────┘
    
    ┌─────────────────────────────────────────────────────────────────────┐
    │ INDIVIDUAL POLICY                                                   │
    │                                                                     │
    │ A policy is 'individual' when:                                     │
    │ • It stands alone (not part of a parent-child list structure)      │
    │ • Has no lettered/numbered sub-items below it                      │
    │ • Not labeled A., B., C. under a parent                            │
    │                                                                     │
    │ For individual policies:                                           │
    │   policy_type = "individual"                                       │
    │   parent_policy_name = None                                        │
    └─────────────────────────────────────────────────────────────────────┘
    
    CLASSIFICATION DECISION TREE:
    
    Is this text labeled A., B., C. (or 1., 2., 3.) in a list?
      ├─ YES → Does a named action/initiative appear above it?
      │   ├─ YES → policy_type = "sub"
      │   │         parent_policy_name = [name of that action]
      │   │         ALSO extract the action header as a "parent" policy
      │   └─ NO  → policy_type = "individual"
      │
      └─ NO → Does this policy have lettered/numbered items below it?
          ├─ YES → policy_type = "parent"
          │         Extract each A/B/C below as "sub" policies
          └─ NO  → policy_type = "individual"
    
    CRITICAL RULES:
    • When you see A., B., C. lists, ALWAYS create both parent and sub policies
    • Extract the parent action/initiative as its own policy
    • Each list item (A, B, C) becomes a sub policy referencing the parent
    • The parent_policy_name should match the action name from section_header
    
    ═══════════════════════════════════════════════════════════════════════
    
    If no policies are found, return an empty list.
    """
    document_text: str = dspy.InputField(
        desc="Text extracted from a climate policy document (NDC, action plan, etc.)"
    )
    document_metadata: DocumentMetadata = dspy.InputField(
        desc="Document name, country, year, and any known section context"
    )
    
    policies: List[ExtractedPolicy] = dspy.OutputField(
        desc="List of extracted policies with correct hierarchy classification"
    )

In [6]:
class PolicyExtractor(dspy.Module):
    """
    Extracts structured policy objects from document text.
    
    Designed to run on chunks/sections of a larger document.
    Handles the full extraction task: identification, extraction, 
    and preliminary soundness assessment.
    """
    
    def __init__(self):
        self.extract = dspy.ChainOfThought(PolicyExtractionSignature)
    
    def forward(self, document_text: str, document_metadata: str = "") -> List[ExtractedPolicy]:
        result = self.extract(
            document_text=document_text,
            document_metadata=document_metadata
        )
        return result.policies

In [7]:
from docling.document_converter import DocumentConverter
converter= DocumentConverter()

In [8]:
document = converter.convert("../pdfs/Chicago-CAP-071822.pdf")

2026-02-17 17:43:58,481 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-17 17:43:58,876 - INFO - Going to convert document batch...
2026-02-17 17:43:58,877 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-02-17 17:43:58,884 - INFO - Loading plugin 'docling_defaults'
2026-02-17 17:43:58,886 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-17 17:43:58,891 - INFO - Loading plugin 'docling_defaults'
2026-02-17 17:43:58,896 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-02-17 17:43:59,987 - INFO - Auto OCR model selected ocrmac.
2026-02-17 17:43:59,991 - INFO - Loading plugin 'docling_defaults'
2026-02-17 17:43:59,993 - INFO - Registered layout engines: ['docling_layout_default', 'docling_experimental_table_crops_layout']
2026-02-17 17:44:02,493 - INFO - Accelerator device: 'mps'
2026-02-17 17:44:03,762 - INFO - Loading plugin 'docling_defaul

In [9]:
markdown_version = document.document.export_to_markdown()

In [10]:
d_metadata = DocumentMetadata(
    country="USA",
    state_or_province="Illinois",
    city="Chicago"
)


In [11]:
policy_extractor = PolicyExtractor()
extracted_policies = policy_extractor(
    document_text=markdown_version,
    document_metadata=d_metadata
)


In [12]:
extracted_policies


[ExtractedPolicy(policy_statement='Reduce Chicago’s greenhouse gas emissions by a minimum of 62% by 2040 from 2017 levels.', verbatim_text="The 2022 CAP aims to chart an equitable path to reduce Chicago's GHG emissions by a minimum of 62% by 2040. ... the City of Chicago aims to reduce the city's level of GHG emissions by 62% by 2040, based on 2017 GHG inventory levels.", policy_type='individual', parent_policy_name=None, section_header="GHG REDUCTION Chicago's TARGETS", sector='Economy-wide', extraction_rationale='Clear quantified, citywide emissions-reduction target with a deadline.'),
 ExtractedPolicy(policy_statement='Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 50% by 2040, prioritizing low- or moderate-income households.', verbatim_text='A. Retrofit residential buildings with 4 or fewer units:20% by 2030 and 50%by2040, prioritizing low- or moder- ate-income households', policy_type='individual', parent_policy_name=None, section_header='Pillar 1 Increase a

In [ ]:
# this version actually ends up over counting lots of policies

In [12]:
import json

with open("extracted_policies_chicago_intermediate_v4", "w") as f:
    json.dump([p.dict() for p in extracted_policies], f, indent=2)
    

/var/folders/f5/bb79s3gs38q_wp4_2kvwbz9w0000gn/T/ipykernel_25125/2420737967.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  json.dump([p.dict() for p in extracted_policies], f, indent=2)
